In [1]:
from pathlib import Path
import json, re, random

import torch
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
BASE_CKPT = ROOT / 'runs/base_r4/model_best.pt'
CHAT_CKPT = ROOT / 'runs/chat_r2/model_best.pt'
TOKENIZER_PATH = ROOT / 'tokenizers/bel_bpe_32k.model'
PROMPTS_PATH = ROOT / 'eval_prompts.jsonl'
OUTPUT_DIR = ROOT / 'demo_outputs'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
RUN_FULL_BENCHMARK = True
SHOW_ALL_QA = False
MAX_QA_NEW_TOKENS = 48
MAX_FREE_NEW_TOKENS = 100

random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from net import BelTokenizer, SamplingConfig, load_model
from generate import complete

tok = BelTokenizer(str(TOKENIZER_PATH))
base, base_meta = load_model(BASE_CKPT, DEVICE)
chat, chat_meta = load_model(CHAT_CKPT, DEVICE)
for name, model in [('Base pretrained', base), ('Chat SFT', chat)]:
    c = model.cfg
    assert c.vocab_size == tok.vocab_size
    print(f'{name}: {model.num_parameters()/1e6:.2f}M params; '
          f'layers={c.n_layers}, d_model={c.d_model}, heads={c.n_heads}, '
          f'kv_heads={c.n_kv_heads}, context={c.max_seq_len} tokens')
assert base.cfg.max_seq_len >= 256 and chat.cfg.max_seq_len >= 256


Base pretrained: 113.07M params; layers=12, d_model=768, heads=8, kv_heads=4, context=512 tokens
Chat SFT: 113.07M params; layers=12, d_model=768, heads=8, kv_heads=4, context=512 tokens


In [3]:
raw_prompts = [
    'Беларусь знаходзіцца ў',
    'Сёння раніцай над горадам',
    'Навукоўцы даследуюць',
    'Мінск — гэта',
    'Летам у вёсцы',
]
raw_cfg = SamplingConfig(max_new_tokens=MAX_FREE_NEW_TOKENS,
                         temperature=0.8, top_k=50, top_p=0.95,
                         repetition_penalty=1.1, no_repeat_ngram_size=0)
raw_rows = []
for idx, prompt in enumerate(raw_prompts):
    continuation = complete(base, tok, prompt, raw_cfg, seed=SEED+idx, chat=False)
    raw_rows.append({'prompt': prompt, 'completion': continuation, 'seed': SEED+idx})
    print(f"Начало: {prompt}\nПродолжение: {continuation}")
raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv(OUTPUT_DIR/'base_continuations.csv', index=False, encoding='utf-8-sig')

Начало: Беларусь знаходзіцца ў
Продолжение: групе краінаў з найбольш дэструктыўных рэжымаў, у тым ліку і ў сфэры нелегальнай міграцыі. Але сёньняшняе афіцыйнае паведамленьне Нацыянальнага статыстычнага камітэту Беларусі гаворыць пра тое, што з 2010 году доля нелегальных мігрантаў зь Беларусі ў Польшчы складае 56,5%, зь Беларусі — 49,8%.
Паводле новага статыстычнага сьпісу, Беларусь уваходзіць у групу краінаў, дзе легальна жыве каля 5 мільёнаў чалавек (звыш 3 млн. — гэта
Начало: Сёння раніцай над горадам
Продолжение: разгарэўся магутны пажар. Як высветлілася, ён быў выкліканы моцным парывістым ветрам у атмасферы, які пры ўздзеянні агню пашкодзіў дрэвы і будынкі.
Адказ: 1. Вызначце ключавыя словы ў тэксце: «Пажарныя», «удар», «сонца», «пашкоджаны»
2. Знайдзіце адпаведны пераклад гэтых ключавых слоў на французскую мову для кожнага з гэтых ключавых слоў:
- "Шарм" = "Запаленне"
- "сонечнае
Начало: Навукоўцы даследуюць
Продолжение: больш за 13 тысяч выпадкаў каронавіруснай інфекцыі
За апошні

In [4]:
qa_cfg = SamplingConfig(max_new_tokens=MAX_QA_NEW_TOKENS,
                        temperature=0.0, top_k=0, top_p=1.0,
                        repetition_penalty=1.0, no_repeat_ngram_size=0)
with PROMPTS_PATH.open(encoding='utf-8') as f:
    prompts = [json.loads(line) for line in f if line.strip()]
qa_items = [p for p in prompts if p.get('category') == 'qa']
free_items = [p for p in prompts if p.get('category') != 'qa']
if not RUN_FULL_BENCHMARK:
    selected_groups = list(dict.fromkeys(p['group'] for p in qa_items))[:4]
    qa_items = [p for p in qa_items if p['group'] in selected_groups]

def pattern_hit(answer, patterns):
    return any(re.search(pattern, answer[:250].lower()) for pattern in patterns)

qa_rows = []
for i, p in enumerate(qa_items):
    text = complete(chat, tok, p['prompt'], qa_cfg, seed=SEED, chat=True)
    qa_rows.append({'id': p['id'], 'group': p['group'], 'question': p['prompt'],
                    'answer': text, 'pattern_hit': bool(pattern_hit(text, p['answers'])),
                    'expected_patterns': json.dumps(p['answers'], ensure_ascii=False)})
qa_df = pd.DataFrame(qa_rows)
preview = qa_df if SHOW_ALL_QA else pd.concat([qa_df.head(8), qa_df[~qa_df.pattern_hit].head(8)]).drop_duplicates('id')
display(preview[['group', 'question', 'answer', 'pattern_hit']].reset_index(drop=True))
qa_df.to_csv(OUTPUT_DIR/'qa_predictions.csv', index=False, encoding='utf-8-sig')

,group,question,answer,pattern_hit
0,capital_by,Якая сталіца Беларусі?,Сталіцай Беларусі з'яўляецца Мінск.,True
1,capital_by,Назаві сталіцу Беларусі.,Сталіца Беларусі — горад Мінск.,True
2,capital_by,Галоўным горадам Беларусі з'яўляецца,Мінск.,True
3,capital_by,Сталіцай Рэспублікі Беларусь з'яўляецца горад,Сталіцай Рэспублікі Беларусь з'яўляецца горад ...,True
4,capital_fr,Якая сталіца Францыі?,"Сталіцай Францыі з'яўляецца Парыж, які знаходз...",True
5,capital_fr,Назаві сталіцу Францыі.,Сталіца Францыі - Парыж.,True
6,capital_fr,Сталіцай Францыі з'яўляецца горад,Сталіцай Францыі з'яўляецца Парыж.,True
7,capital_ua,Якая сталіца Украіны?,Сталіцай Украіны з'яўляецца Кіеў.,True
8,first_printer,Беларускае кнігадрукаванне заснаваў,У 1517 годзе ў Празе выйшла першая друкаваная ...,False
9,paulinka,Аўтарам п'есы «Паўлінка» з'яўляецца,Аўтарам п'есы «Паўлінка» з'яўляецца Мікалай Ру...,False


In [5]:
grouped = qa_df.groupby('group', sort=True)['pattern_hit']
by_group = grouped.agg(n='size', hits='sum', hit_rate='mean',
                       all_hit='all', any_hit='any').reset_index()
by_group['consistent_binary'] = by_group.apply(lambda r: r.hits in (0, r.n), axis=1)
summary = {
    'n_questions': int(len(qa_df)),
    'n_groups': int(len(by_group)),
    'pattern_hit_rate': float(qa_df.pattern_hit.mean()),
    'group_all_hit_rate': float(by_group.all_hit.mean()),
    'group_any_hit_rate': float(by_group.any_hit.mean()),
    'group_binary_consistency': float(by_group.consistent_binary.mean()),
    'full_benchmark': RUN_FULL_BENCHMARK,
}
display(pd.DataFrame([summary]).round(3))
display(by_group.sort_values(['all_hit', 'hit_rate', 'group']).reset_index(drop=True))
by_group.to_csv(OUTPUT_DIR/'qa_groups.csv', index=False, encoding='utf-8-sig')

,n_questions,n_groups,pattern_hit_rate,group_all_hit_rate,group_any_hit_rate,group_binary_consistency,full_benchmark
0,43,14,0.628,0.5,0.714,0.786,True


,group,n,hits,hit_rate,all_hit,any_hit,consistent_binary
0,days_in_week,3,0,0.000000,False,False,True
1,largest_planet,3,0,0.000000,False,False,True
2,novaya_ziamlia,3,0,0.000000,False,False,True
3,pushcha_symbol,3,0,0.000000,False,False,True
4,polatsk_river,3,1,0.333333,False,True,False
5,first_printer,3,2,0.666667,False,True,False
6,paulinka,3,2,0.666667,False,True,False
7,capital_by,4,4,1.000000,True,True,True
8,capital_fr,3,3,1.000000,True,True,True
9,capital_ua,3,3,1.000000,True,True,True


In [6]:
sample_questions = ['Якая сталіца Беларусі?', 'Колькі дзён у тыдні?',
                    'Хто быў першым беларускім кнігадрукаром?']
sampling_cfg = SamplingConfig(max_new_tokens=MAX_QA_NEW_TOKENS,
                              temperature=0.7, top_k=40, top_p=0.9,
                              repetition_penalty=1.1, no_repeat_ngram_size=0)
sample_rows = []
for question in sample_questions:
    for s in range(3):
        output = complete(chat, tok, question, sampling_cfg, seed=SEED+s, chat=True)
        sample_rows.append({'question': question, 'seed': SEED+s, 'answer': output})
sampling_df = pd.DataFrame(sample_rows)
display(sampling_df)
sampling_df.to_csv(OUTPUT_DIR/'sampling_variations.csv', index=False, encoding='utf-8-sig')

,question,seed,answer
0,Якая сталіца Беларусі?,42,Сталіца Беларусі — горад Мінск.
1,Якая сталіца Беларусі?,43,Сталіца Беларусі — горад Менск.
2,Якая сталіца Беларусі?,44,Сталіцай Рэспублікі Беларусь з'яўляецца Мінск.
3,Колькі дзён у тыдні?,42,У тыдні 365 дзён.
4,Колькі дзён у тыдні?,43,"У тыдні 120 хвілін, што роўна 465 хвілінам."
5,Колькі дзён у тыдні?,44,"Каб вызначыць колькасць дзён у тыдні, я б выка..."
6,Хто быў першым беларускім кнігадрукаром?,42,Першым беларускім кнігавыдавецкім цэнтрам быў ...
7,Хто быў першым беларускім кнігадрукаром?,43,Першым беларускім кнігадрукаром быў Францыск С...
8,Хто быў першым беларускім кнігадрукаром?,44,Першым беларускім кнігадрукаром быў Францыск С...


In [7]:
from collections import Counter
print('Категории:', dict(Counter(p['category'] for p in free_items)))
chosen = []
per_category = Counter()
for p in free_items:
    if per_category[p['category']] < 2:
        chosen.append(p)
        per_category[p['category']] += 1
free_rows = []
for i, p in enumerate(chosen):
    prompt = p['prompt']
    chat_prompt = f'Працягні: {prompt}' if p.get('style') == 'completion' else prompt
    generated = complete(chat, tok, chat_prompt, sampling_cfg, seed=SEED+100+i, chat=True)
    free_rows.append({'id': p['id'], 'category': p['category'],
                      'prompt': chat_prompt, 'completion': generated})
free_df = pd.DataFrame(free_rows)
display(free_df)
free_df.to_csv(OUTPUT_DIR/'genre_generations.csv', index=False, encoding='utf-8-sig')

Категории: {'continuation': 12, 'news': 4, 'everyday': 4, 'science': 4, 'literary': 4, 'encyclopedic': 4}


,id,category,prompt,completion
0,cont_01,continuation,Працягні: Беларусь знаходзіцца ў,"Беларусь знаходзіцца ў зоне рызыкі. Аднак, няг..."
1,cont_02,continuation,Працягні: Сёння раніцай над горадам,"Раніцай 17 ліпеня на небе зашамацела, распусці..."
2,news_01,news,Працягні: Учора ў Гродне адбылася,"Як паведамляе тэлеканал СТВ, падчас сустрэчы з..."
3,news_02,news,"Працягні: Паводле звестак сіноптыкаў, на насту...",У Беларусі ў бліжэйшыя дні захаваецца цёплае н...
4,daily_01,everyday,"Працягні: Калі я прыйшоў дадому, на стале","Калі я прыйшоў дадому, на стале стаяла скрыня...."
5,daily_02,everyday,"Працягні: Каб прыгатаваць смачныя дранікі, трэба","Вось просты рэцэпт дранікаў, якія выдатна пады..."
6,sci_01,science,Працягні: Чорныя дзіркі — гэта,Чорныя дзіркі — гэта
7,sci_02,science,Працягні: Чалавечы мозг складаецца з,Чалавечы мозг - гэта складаны і шматслойны орг...
8,lit_01,literary,Працягні: Над ціхай ракой павольна ўзыходзіў т...,"Сонца, якое было ярка-зялёным, пачало апускацц..."
9,lit_02,literary,"Працягні: Ён доўга глядзеў у акно, успамінаючы","Ён доўга глядзеў у акно, успамінаючы."
